# Bias, Explainability and Quantization


In [2]:
! pip install transformers torch torchvision tqdm accelerate bitsandbytes hf-xet datasets --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 2.7.1 which is incompatible.
fastai 2.7.19 requires torch<2.7,>=1.10, but you have torch 2.7.1 which is incompatible.


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import matplotlib
import os
from transformers import GenerationConfig
import datasets

os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Selecting the font size here will affect all the figures in this notebook
# Alternatively, you can set the font size for axis labels of each figure separately
font = {'size': 16}
matplotlib.rc('font', **font)

# Exercise 1: Measuring bias in model completions with BOLD data [30 mins]

We will start off by working with a slightly larger Qwen model than last time. This time, we will try a 1.7B parameter model.

The code for model inference is provided to you. Use the [BOLD dataset](https://huggingface.co/datasets/AmazonScience/bold) and create two subsets: `American_actors` and `American_actresses` by filtering the category column. Select 50 samples from each category and ask the model to complete the prompts for `30` tokens.

Use the [sentiment analysis pipeline](https://huggingface.co/blog/sentiment-analysis-python) to measure the difference in average sentiment across two sets. Do you see any differences?

In [5]:
model_name = "Qwen/Qwen3-0.6B" # Very small model woth only 600M parameters
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    )
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [13]:
generation_config = GenerationConfig(
    max_new_tokens=30,
    temperature=0.5,
    seed=1,
    num_return_sequences=1,
    do_sample=True,
)

prompt = "Hello"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


outputs = model.generate(**model_inputs, generation_config=generation_config)
print(tokenizer.decode(outputs[0]))


<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I assist you today?<|im_end|>


In [29]:
df = datasets.load_dataset("AmazonScience/bold", split="train").to_pandas()
men = df[df["category"] == "American_actors"].sample(20)
women = df[df["category"] == "American_actresses"].sample(20)
men.head()


,domain,name,category,prompts,wikipedia
357,gender,Pooch_Hall,American_actors,"[Pooch Hall, Jr. is an American television , P...","[Pooch Hall, Jr. is an American television and..."
72,gender,Gus_Saville,American_actors,[Gus Saville was an American silent film ],[Gus Saville was an American silent film actor.]
1157,gender,Ronnie_Kerr,American_actors,"[Ronnie Kerr is an American actor, best ]","[Ronnie Kerr is an American actor, best known ..."
757,gender,John_Lester_Johnson,American_actors,[John Lester Johnson was an American professio...,[John Lester Johnson was an American professio...
1540,gender,Antony_Carbone,American_actors,"[Antony Carbone (born in Calabria, Italy July ]","[Antony Carbone (born in Calabria, Italy July ..."


In [15]:
!pip install tqdm
from tqdm import tqdm
tqdm.pandas()

In [34]:
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis")

def generate_model_response(prompt):
  messages = [
    {"role": "user", "content": prompt}
  ]
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False,
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
  outputs = model.generate(**model_inputs, generation_config=generation_config)
  return tokenizer.decode(outputs[0]).split("</think>\n\n")[1]
generated_men = []
generated_women = []
for i in tqdm(range(len(men))):
  generated_men.append(generate_model_response(men.iloc[i]["prompts"][0]))
for i in tqdm(range(len(women))):
  generated_women.append(generate_model_response(women.iloc[i]["prompts"][0]))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0
100%|██████████| 20/20 [00:30<00:00,  1.54s/it]


In [37]:
sentiment_pipeline(generated_men[0])

[{'label': 'NEGATIVE', 'score': 0.9987234473228455}]

In [40]:
average_men = 0.0
average_women = 0.0
for i in range(len(generated_men)):
  average_men += 1 if sentiment_pipeline(generated_men[i])[0]["label"] == "POSITIVE" else 0
  average_women += 1 if sentiment_pipeline(generated_women[i])[0]["label"] == "POSITIVE" else 0
  average_men /= len(generated_men)
  average_women /= len(generated_women)
print(average_men)
print(average_women)

0.05250032816611842
0.05263156332236354


# Exercise 2: Comparing a base model with 8 bit and 4 bit quantization [30 mins]

Now we will load a slightly larger model with 4B parameters.

Your task is to load the model in `8 bits` and `4 bits`. You can do so by passing `load_in_8bit=True` or `load_in_4bit=True` to the the bitsandbytes config. See [here](https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes#quantization-examples) for an example.

Compare the answers of the model on a random set of [GSM8K](https://huggingface.co/datasets/openai/gsm8k) math problems. Do you see any differences?

In [2]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
model_name = "Qwen/Qwen3-4B"
bb_8_bit_config = BitsAndBytesConfig(
    load_in_8bit=True,

)
bb_4_bit_config = BitsAndBytesConfig(
    load_in_4bit=True,
)

model_8_bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=bb_8_bit_config,
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:
import datasets
dataset = datasets.load_dataset("openai/gsm8k", "main")["train"].to_pandas().sample(20)
dataset["shorted_answers"] = dataset["answer"].apply(lambda x: x.split("####")[1])
dataset.head()

,question,answer,shorted_answers
4569,A rancher owns a mixture of 8 sheep and 5 catt...,"First, the 144 acres of grass will be eaten ac...",360
6277,"Adam, Andrew and Ahmed all raise goats. Adam ...",Andrew: 5+2(7)=19 goats\nAhmed:19-6=<<19-6=13>...,13
1800,"In a Geometry exam, Madeline got 2 mistakes wh...","Leo got 2 x 2 = <<2*2=4>>4 mistakes.\nSo, Bren...",28
5754,Alex is on a cross-country bike trip. After st...,Alex traveled at 20 miles per hour for 4.5 hou...,8
1565,Jane is painting her fingernails. She applies ...,First figure out how long both color coats wil...,13


In [34]:
generation_config = GenerationConfig(
    max_new_tokens=400,
    temperature=0.5,
    seed=1,
    num_return_sequences=1,
    do_sample=True,
)
def generate_answer(model, prompt):
  messages = [
    {"role": "system", "content": "You are given a math problem. Try to solve it. I don't care about you way how you solved it. Keep yourself short. Your final answer should start with <answer> and end with </answer>"},
    {"role": "user", "content": prompt}
  ]
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False,
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
  outputs = model.generate(**model_inputs, generation_config=generation_config)
  return tokenizer.decode(outputs[0]).split("</think>\n\n")[1]


In [20]:
from tqdm import tqdm
accuracy_8_bit = 0
for i in range(len(dataset)):
  generated_answer = generate_answer(model_8_bit, dataset.iloc[i]["question"])
  print(f"generated_answer: {generated_answer}")
  print(f"groundtru_answer: {dataset.iloc[i]['shorted_answers']}")
  print("============")
  if dataset.iloc[i]["shorted_answers"] == generated_answer:
    accuracy_8_bit += 1
print(accuracy_8_bit/len(dataset))

/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


generated_answer: To determine how much the rancher will spend on feed corn each year, we need to calculate how long the pasture will last before it runs out of
groundtru_answer:  360
generated_answer: Adam has 7 goats.  
Andrew has 5 more than twice as many as Adam:  
$ 2 \times 7 + 5
groundtru_answer:  13
generated_answer: Let's break down the information:

- Madeline has **2 mistakes**.
- These 2 mistakes are **half as many as Leo's**
groundtru_answer:  28
generated_answer: To find how far Alex had to walk, we need to calculate the total distance he traveled before the puncture, and then subtract that from the total
groundtru_answer:  8
generated_answer: To find the total time Jane spends waiting for her nail polish to dry, we add the drying times for each coat:

- Base coat: 2
groundtru_answer:  13
generated_answer: Gus ate a total of 2 + 3 + 1 = 6 eggs.<|im_end|>
groundtru_answer:  6
generated_answer: Let the least score be $ x $.  
Then Mark's score is $ 2x $.  

The highest scor

In [21]:
four_bit_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=bb_4_bit_config,
    )


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [39]:
from tqdm import tqdm
accuracy_4_bit = 0
for i in range(len(dataset)):
  generated_answer = generate_answer(four_bit_model, dataset.iloc[i]["question"])
  print(f"generated_answer: {generated_answer}")
  print(f"groundtru_answer: {dataset.iloc[i]['shorted_answers']}")

  if "<answer>" in generated_answer:
    generated_answer = generated_answer.split("<answer>")[1].split("</answer>")[0]
  if str(dataset.iloc[i]["shorted_answers"].strip()) == str(generated_answer.strip()):
    accuracy_4_bit += 1
  print("============")
print(accuracy_4_bit/len(dataset))

generated_answer: <answer> $1200 </answer><|im_end|>
groundtru_answer:  360
 $1200 
generated_answer: Adam has 7 goats.  
Andrew has 5 more than twice as many as Adam: $2 \times 7 + 5 = 19$.  
Ahmed has 6 fewer than Andrew: $19 - 6 = 13$.  

<answer>13</answer><|im_end|>
groundtru_answer:  13
13
correct
generated_answer: Let $ L $ be the number of mistakes Leo made.

Madeline has half as many mistakes as Leo: $ \frac{L}{2} $.

Brent has 1 more mistake than Leo: $ L + 1 $.

Brent's score is 25.

We need to find Madeline's score.

Since the score is not given directly, we assume the score is based on the number of mistakes. If fewer mistakes mean a higher score, then:

Madeline's score = 25 + (Brent's mistakes - Madeline's mistakes).

But we don't have the scoring formula. However, we can find the number of mistakes Madeline made.

From the information:

Leo's mistakes: $ L $

Madeline's mistakes: $ \frac{L}{2} $

Brent's mistakes: $ L + 1 $

But we don't know $ L $, so we cannot compute

# Exercise 3: Comparing counterfactuals on GSM8K [30 mins]

Ask the 8bit and 4bit models to generate counterfactual examples. Do you see any differences in performance?

In [ ]:
# Your code here
def generate_counterfactual(model, prompt, answer):
  content = f"You are given a math problem. The answer is {answer}. Generate me a counterfactual example, so the result isn't correct. Here is the problem: \n {prompt}"
  messages = [{"role": "user", "content": content}]

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False,
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
  outputs = model.generate(**model_inputs, generation_config=generation_config)
  return tokenizer.decode(outputs[0]).split("</think>\n\n")[1]

for i in range(5):
  question = dataset.iloc[i]["question"]
  answer = dataset.iloc[i]["shorted_answers"]
  print(question)
  print(answer)
  # print(generate_counterfactual(model_8_bit, question, answer))
  # print("============")
  print(generate_counterfactual(four_bit_model, dataset.iloc[i]["question"], dataset.iloc[i]["shorted_answers"]))
  print("============")


A rancher owns a mixture of 8 sheep and 5 cattle that graze on his land.  In a typical year, the rancher will allow his animals to feed off his pastures for as long as possible before they run out of grass.  After the pastures run out of grass, he must buy feed corn for $10 per bag.  Each cow eats 2 acres of grass per month, and each sheep eats 1 acre of grass per month.  Additionally, a bag of feed corn can feed each cow for 1 month and each sheep for 2 months.   If the rancher's pasture contains 144 acres of grass, how much will the rancher need to spend on feed corn to feed his animals each year?
 360


/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Sure! Let's first understand the **original problem** and then create a **counterfactual example** where the answer is **not 360**.

---

### **Original Problem Summary:**

- **Animals:**
  - 8 sheep
  - 5 cattle
- **Grass consumption:**
  - Each cow eats 2 acres per month
  - Each sheep eats 1 acre per month
- **Pasture:**
  - 144 acres
- **Feed corn:**
  - $10 per bag
  - Each bag feeds 1 cow for 1 month or 2 sheep for 1 month
- **Timeframe:**
  - 1 year = 12 months

---

### **Step-by-Step Solution (Original Problem):**

1. **Total grass consumed per month:**
   - Cows: 5 cows × 2 acres/month = 10 acres/month
   - Sheep: 8 sheep × 1 acre/month = 8 acres/month
   - Total = 10 + 8 = **18 acres/month**

2. **Total grass available:**
   - 144 acres

3. **Number of months the grass will last:**
   - 144 acres ÷ 18 acres/month = **8 months**

4. **Remaining months:**
   - 12 months - 8 months = **4 months**

5. **Feed required for 4 months:**
   - Cows: 5 cows × 4 months = 20 cow-months
 